# Functional Context of the Vitamin D Transcriptional Signature

This notebook provides functional context for the Vitamin D transcriptional
signature by annotating and performing targeted enrichment analysis on genes
that were stably selected by an interpretable ElasticNet model.

**Important note:** This analysis is used for contextual interpretation
of transcriptomic patterns and does not establish causal mechanisms.

In [ ]:
import numpy as np
import pandas as pd

## Inputs

This notebook is run standalone and relies on exported results from
previous analyses.

Required inputs:
- `stable_genes_elasticnet_core_score.csv`: stable genes identified by ElasticNet
- `geneinfo_beta.txt`: gene annotation table from LINCS (gene_id → gene_symbol)

In [ ]:
stable_genes = pd.read_csv(
    "../data/exports/stable_genes_elasticnet_core_score.csv"
)

gene_info = pd.read_csv(
    "../data/raw_data/geneinfo_beta.txt",
    sep="\t"
)

gene_info = gene_info[["gene_id", "gene_symbol", "gene_title"]]

In [ ]:
display(stable_genes.head())
display(gene_info.head())

In [ ]:
stable_genes_annot = stable_genes.merge(
    gene_info,
    on="gene_id",
    how="left"
)

print(stable_genes_annot.head(10))


In [ ]:
# --- Clean and standardize stable_genes_annot ---

# Prefer gene symbols/titles coming from geneinfo (suffix _y)
# and drop duplicated columns from previous merges

cols_to_keep = {
    "gene_id": "gene_id",
    "selection_freq": "selection_freq",
    "sign_consistency": "sign_consistency",
    "mean_abs_coef": "mean_abs_coef",
    "gene_symbol_y": "gene_symbol",
    "gene_title_y": "gene_title",
}

stable_genes = (
    stable_genes_annot[list(cols_to_keep.keys())]
    .rename(columns=cols_to_keep)
    .sort_values("mean_abs_coef", ascending=False)
    .reset_index(drop=True)
)

display(stable_genes.head(10))


In [ ]:
# Use genes with strong stability criteria
# (You can adjust thresholds, but keep them fixed once chosen for reproducibility.)
ENRICH_MIN_FREQ = 3
ENRICH_REQUIRE_SIGN = True

stable_genes_annot = stable_genes.copy()

mask = stable_genes_annot["selection_freq"] >= ENRICH_MIN_FREQ
if ENRICH_REQUIRE_SIGN:
    mask &= (stable_genes_annot["sign_consistency"] == 1.0)

gene_list = (
    stable_genes_annot.loc[mask, "gene_symbol"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Number of genes for enrichment:", len(gene_list))
print("First 10 symbols:", gene_list[:10])


## Targeted functional enrichment (contextual)

We run a targeted enrichment analysis on the stable gene subset to obtain
high-level functional context (e.g., GO Biological Process, Reactome).

This is not used for pathway discovery, but to summarize recurring biological
themes supported by the gene-level results.


In [ ]:
from pathlib import Path

out_dir = Path("../data/exports/functional_context")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "stable_gene_symbols_for_enrichment.csv"
pd.Series(gene_list, name="gene_symbol").to_csv(out_path, index=False)

print("Saved gene list to:", out_path)


## Functional enrichment (external tools)

To obtain functional context, the list of stable gene symbols was exported
and analyzed using external enrichment tools (e.g., g:Profiler or Enrichr).

Only high-level functional categories were considered to summarize
recurring biological themes. This step is used for contextual interpretation
and does not aim to establish causal mechanisms.

In [ ]:
enrich_path = "../data/exports/functional_context/enrichment_results.csv"

enrich = pd.read_csv(enrich_path)

# Keep only the most informative columns (adjust names if needed)
cols = [c for c in enrich.columns if c.lower() in [
    "source", "name", "p_value", "adjusted_p_value",
    "term_size", "intersection_size"
]]

enrich_clean = enrich[cols].copy()

display(enrich_clean.head(10))

## Summary (functional context)

Functional enrichment analysis of the stable gene subset identified by the
ElasticNet model reveals significant enrichment in broad biological
processes related to cellular regulation.

Enriched GO Biological Process terms are characterized by large gene sets
and substantial overlap with the stable gene list, suggesting that the
Vitamin D transcriptional signature reflects a distributed regulatory
program rather than a single, narrowly defined pathway.

These results provide functional context that is fully consistent with the
gene-level and modeling analyses, and should be interpreted as associative
transcriptomic patterns rather than direct evidence of specific molecular
mechanisms.